# PDF Processing Pipeline: Complete Workflow

Steps:
1. Extract layout from original PDF → JSON
2. Reconstruct tables from captions
3. Mask tables/figures in PDF → Masked PDF
4. Extract layout from masked PDF → JSON ✨
5. **Use masked PDF JSON for text extraction** ✨
6. Stitch text paragraphs
7. Save outputs

**Date**: 2025-12-31

## Setup & Imports

In [11]:
import sys
import json
from pathlib import Path
import fitz
from IPython.display import Image, display
import pandas as pd
import importlib.util
from collections import Counter
from datetime import datetime

project_root = Path.cwd()
sys.path.insert(0, str(project_root))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

mask_tables = load_module('mask_tables', project_root / 'scripts/docling/mask_tables.py')
visualize = load_module('visualize', project_root / 'scripts/visualize_docling_full.py')
text_proc = load_module('text_proc', project_root / 'parsers/text_processing.py')

process_pdf_with_masking = mask_tables.process_pdf_with_masking
reconstruct_tables_from_lists = visualize.reconstruct_tables_from_lists
ContextAwareStitcher = text_proc.ContextAwareStitcher
remove_citations = text_proc.remove_citations

print("✅ Imports successful!")

✅ Imports successful!


## Step 1: Configure Paths

In [12]:
PDF_PATH = Path('files/organized_pdfs/PMC1448691_his_2369.pdf')
PMCID = 'PMC1448691'

DOCLING_OUTPUT_DIR = Path('out/docling_full')
MASKED_PDF_DIR = Path('out/masked_pdfs')
TEXT_OUTPUT_DIR = Path('out/text')
TABLES_DIR = Path('files/tables')
FIGURES_DIR = Path('files/figures')

for d in [DOCLING_OUTPUT_DIR, MASKED_PDF_DIR, TEXT_OUTPUT_DIR, TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📄 PDF: {PDF_PATH.name}")
print(f"📁 PMCID: {PMCID}")

📄 PDF: PMC1448691_his_2369.pdf
📁 PMCID: PMC1448691


## Step 2: Extract Layout from Original PDF

In [13]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = False
pipeline_options.do_ocr = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

print(f"🔄 Extracting layout from ORIGINAL PDF...")
result = converter.convert(str(PDF_PATH))
doc = result.document

all_elements = []
for element, level in doc.iterate_items():
    label = str(getattr(element, "label", "UNKNOWN")).split('.')[-1].upper()
    if not (hasattr(element, 'prov') and element.prov):
        continue
    prov = element.prov[0]
    bbox = prov.bbox
    text = ""
    if hasattr(element, 'text'):
        text = element.text
    elif hasattr(element, 'caption') and element.caption:
        text = element.caption.text
    all_elements.append({
        "type": label,
        "page": prov.page_no,
        "level": level,
        "bbox": {"x1": bbox.l, "y1": bbox.t, "x2": bbox.r, "y2": bbox.b},
        "text": text.strip() if text else None
    })

docling_json_path = DOCLING_OUTPUT_DIR / f"{PDF_PATH.stem}_full_layout.json"
with open(docling_json_path, 'w') as f:
    json.dump({
        "metadata": {"pdf_path": str(PDF_PATH), "tool": "Docling", "extraction_date": datetime.now().isoformat()},
        "page_dimensions": {no: {"width": p.size.width, "height": p.size.height} for no, p in doc.pages.items()},
        "elements": all_elements
    }, f, indent=2)

types = Counter([el['type'] for el in all_elements])
print(f"\n✅ Original PDF: {len(all_elements)} elements")
print(f"   Tables: {types.get('TABLE', 0)}")
print(f"   Figures: {types.get('PICTURE', 0)}")
print(f"   Captions: {types.get('CAPTION', 0)}")
print(f"💾 {docling_json_path}")

INFO: detected formats: [<InputFormat.PDF: 'pdf'>]
INFO: Going to convert document batch...
INFO: Initializing pipeline for StandardPdfPipeline with options hash 06b2cfbc41861ca6858eec36b7d53267
INFO: Auto OCR model selected ocrmac.
INFO: Accelerator device: 'mps'


🔄 Extracting layout from ORIGINAL PDF...


INFO: Processing document PMC1448691_his_2369.pdf
INFO: Finished converting document PMC1448691_his_2369.pdf in 38.51 sec.



✅ Original PDF: 422 elements
   Tables: 2
   Figures: 9
   Captions: 14
💾 out/docling_full/PMC1448691_his_2369_full_layout.json


## Step 3: Reconstruct Tables

In [14]:
print("🔄 Reconstructing tables...")
reconstructed_elements = reconstruct_tables_from_lists(str(docling_json_path))
reconstructed_tables = [el for el in reconstructed_elements if el.get('type') == 'RECONSTRUCTED_TABLE']
print(f"✅ Created {len(reconstructed_tables)} reconstructed tables")

🔄 Reconstructing tables...
✅ Created 4 reconstructed tables


## Step 4: Mask Tables & Figures

In [15]:
print("🔄 Masking tables and figures...")
masked_pdf_path, _, masked_elements = process_pdf_with_masking(
    pdf_path=PDF_PATH,
    json_path=docling_json_path,
    output_dir=MASKED_PDF_DIR
)
print(f"\n✅ Masked PDF: {masked_pdf_path}")
print(f"🚫 Masked: {len(masked_elements)} elements (tables/figures/captions)")

INFO: Processing: PMC1448691_his_2369.pdf
INFO:   Reconstructing tables from elements...
INFO:   Found 29 maskable elements (including 4 reconstructed tables)
INFO:     ✓ Masked element CAPTION with text = Table 1. B-cell cutaneous lymphoma. Learning from the Workshop on page 2
INFO:     ✓ Masked element RECONSTRUCTED_TABLE with text = NO_TEXT on page 2
INFO:     ✓ Masked element PICTURE with text = None on page 3
INFO:     ✓ Masked element CAPTION with text = Figure 1. a-f, Primary cutaneous follicle centre lymphoma. a, Nodular pattern. b, Centroblastic predominance. c, CD20. d, CD10. e, Bcl-2. f, Ki67. g-j, Primary cutaneous diffuse large B-cell lymphoma, leg type. g,h, morphology, H&E. i, CD20. j, Ki67. Cases contributed by C. Girardet (A6) and R. S. Robertorye (A8). on page 3
INFO:     ✓ Masked element TABLE with text = None on page 5
INFO:     ✓ Masked element CAPTION with text = Table 2. Comparison of the immunophenotype of normal and tumoral plasmacytoid dendritic cells (PDC) on

🔄 Masking tables and figures...


INFO:   ✓ Saved masked PDF: out/masked_pdfs/PMC1448691_his_2369_masked.pdf
INFO:   ✓ Masked 29 regions
INFO:   ✓ Extracted 379 text elements




✅ Masked PDF: out/masked_pdfs/PMC1448691_his_2369_masked.pdf
🚫 Masked: 29 elements (tables/figures/captions)


## Step 5: Extract Layout from Masked PDF ✨

**Extract clean text elements from the masked PDF**

In [16]:
print(f"🔄 Extracting layout from MASKED PDF...")
result_masked = converter.convert(str(masked_pdf_path))
doc_masked = result_masked.document

masked_pdf_elements = []
for element, level in doc_masked.iterate_items():
    label = str(getattr(element, "label", "UNKNOWN")).split('.')[-1].upper()
    if not (hasattr(element, 'prov') and element.prov):
        continue
    prov = element.prov[0]
    bbox = prov.bbox
    text = ""
    if hasattr(element, 'text'):
        text = element.text
    elif hasattr(element, 'caption') and element.caption:
        text = element.caption.text
    masked_pdf_elements.append({
        "type": label,
        "page": prov.page_no,
        "level": level,
        "bbox": {"x1": bbox.l, "y1": bbox.t, "x2": bbox.r, "y2": bbox.b},
        "text": text.strip() if text else None
    })

masked_json_path = DOCLING_OUTPUT_DIR / f"{masked_pdf_path.stem}_full_layout.json"
with open(masked_json_path, 'w') as f:
    json.dump({
        "metadata": {"pdf_path": str(masked_pdf_path), "tool": "Docling", "extraction_date": datetime.now().isoformat()},
        "page_dimensions": {no: {"width": p.size.width, "height": p.size.height} for no, p in doc_masked.pages.items()},
        "elements": masked_pdf_elements
    }, f, indent=2)

masked_types = Counter([el['type'] for el in masked_pdf_elements])
print(f"\n✅ Masked PDF: {len(masked_pdf_elements)} elements")
print(f"\n📊 COMPARISON:")
print(f"   Original PDF: {len(all_elements)} elements")
print(f"   Masked PDF:   {len(masked_pdf_elements)} elements")
print(f"   Removed:      {len(all_elements) - len(masked_pdf_elements)} elements")
print(f"\n🔍 Masked PDF element types:")
for t, c in masked_types.most_common():
    print(f"   {t}: {c}")
print(f"\n💾 {masked_json_path}")

# Extract text elements from masked PDF for stitching
text_element_types = {'TEXT', 'PARAGRAPH', 'SECTION_HEADER', 'TITLE', 'LIST', 'LIST_ITEM'}
text_elements = [el for el in masked_pdf_elements if el.get('type') in text_element_types]
print(f"\n📝 Text elements for processing: {len(text_elements)}")

INFO: detected formats: [<InputFormat.PDF: 'pdf'>]
INFO: Going to convert document batch...
INFO: Processing document PMC1448691_his_2369_masked.pdf


🔄 Extracting layout from MASKED PDF...


INFO: Finished converting document PMC1448691_his_2369_masked.pdf in 21.89 sec.



✅ Masked PDF: 284 elements

📊 COMPARISON:
   Original PDF: 422 elements
   Masked PDF:   284 elements
   Removed:      138 elements

🔍 Masked PDF element types:
   LIST_ITEM: 153
   TEXT: 111
   SECTION_HEADER: 19
   FOOTNOTE: 1

💾 out/docling_full/PMC1448691_his_2369_masked_full_layout.json

📝 Text elements for processing: 283


## Step 6: Stitch Text ✨

**Using text elements from the masked PDF JSON**

In [17]:
stitcher = ContextAwareStitcher()
paragraphs = [remove_citations(el.get('text', '')) for el in text_elements if el.get('text', '').strip()]
stitched = stitcher.reconstruct_paragraphs(paragraphs)
print(f"✂️ Stitching: {len(paragraphs)} → {len(stitched)} paragraphs")
print(f"   Merged: {len(paragraphs) - len(stitched)} split paragraphs")

✂️ Stitching: 283 → 230 paragraphs
   Merged: 53 split paragraphs


## Step 7: Save Text

In [18]:
text_path = TEXT_OUTPUT_DIR / f"{PMCID}_stitched.txt"
with open(text_path, 'w') as f:
    f.write(f"Document: {PMCID}\n{'='*80}\n\n")
    for p in stitched:
        if p.strip():
            f.write(f"{p}\n\n")
print(f"💾 {text_path} ({text_path.stat().st_size / 1024:.1f} KB)")

💾 out/text/PMC1448691_stitched.txt (85.6 KB)


## Step 8: Crop Images from Original PDF

In [19]:
table_data, figure_data = [], []
tc, fc = 1, 1

for el in masked_elements:
    t = el.get('type')
    if t in ['TABLE', 'RECONSTRUCTED_TABLE']:
        table_data.append({'table_id': str(tc), 'caption': el.get('text', f'Table {tc}'),
                          'page': el.get('page'), 'bbox': el.get('bbox'), 'type': t.lower()})
        tc += 1
    elif t in ['FIGURE', 'PICTURE']:
        figure_data.append({'figure_id': str(fc), 'caption': el.get('text', f'Figure {fc}'),
                           'page': el.get('page'), 'bbox': el.get('bbox'), 'type': t.lower()})
        fc += 1

doc = fitz.open(str(PDF_PATH))
for t in table_data:
    if t['page'] and t['bbox']:
        p = doc[t['page'] - 1]
        h = p.rect.height
        b = t['bbox']
        r = fitz.Rect(b['x1'], h - max(b['y1'], b['y2']), b['x2'], h - min(b['y1'], b['y2']))
        pix = p.get_pixmap(clip=r, matrix=fitz.Matrix(2, 2))
        path = TABLES_DIR / f"{PMCID}_table_{t['table_id']}.png"
        pix.save(str(path))
        t['image_path'] = str(path)

for f in figure_data:
    if f['page'] and f['bbox']:
        p = doc[f['page'] - 1]
        h = p.rect.height
        b = f['bbox']
        r = fitz.Rect(b['x1'], h - max(b['y1'], b['y2']), b['x2'], h - min(b['y1'], b['y2']))
        pix = p.get_pixmap(clip=r, matrix=fitz.Matrix(2, 2))
        path = FIGURES_DIR / f"{PMCID}_figure_{f['figure_id']}.png"
        pix.save(str(path))
        f['image_path'] = str(path)

doc.close()
print(f"🖼️ Cropped {len(table_data)} tables, {len(figure_data)} figures")

🖼️ Cropped 6 tables, 9 figures


## Step 9: Save Metadata

In [20]:
if table_data:
    with open(TABLES_DIR / f"{PMCID}_tables.json", 'w') as f:
        json.dump(table_data, f, indent=2)
if figure_data:
    with open(FIGURES_DIR / f"{PMCID}_figures.json", 'w') as f:
        json.dump(figure_data, f, indent=2)
print("💾 Metadata saved")

💾 Metadata saved


## Summary

In [21]:
print("="*80)
print("📊 PIPELINE COMPLETE")
print("="*80)
print(f"\n📄 Input: {PDF_PATH.name}")
print(f"\n📊 Processing:")
print(f"   Original elements:  {len(all_elements)}")
print(f"   Masked PDF elements: {len(masked_pdf_elements)}")
print(f"   Text elements used:  {len(text_elements)}")
print(f"   Stitched paragraphs: {len(stitched)}")
print(f"\n📝 Outputs:")
print(f"   Original JSON:     {docling_json_path.name}")
print(f"   Masked PDF:        {masked_pdf_path.name}")
print(f"   Masked PDF JSON:   {masked_json_path.name} ✨")
print(f"   Text file:         {text_path.name}")
print(f"   Tables:            {len(table_data)} images")
print(f"   Figures:           {len(figure_data)} images")
print("\n✅ Done!")
print("\n💡 Text elements came from masked PDF JSON (clean, no tables/figures)")

📊 PIPELINE COMPLETE

📄 Input: PMC1448691_his_2369.pdf

📊 Processing:
   Original elements:  422
   Masked PDF elements: 284
   Text elements used:  283
   Stitched paragraphs: 230

📝 Outputs:
   Original JSON:     PMC1448691_his_2369_full_layout.json
   Masked PDF:        PMC1448691_his_2369_masked.pdf
   Masked PDF JSON:   PMC1448691_his_2369_masked_full_layout.json ✨
   Text file:         PMC1448691_stitched.txt
   Tables:            6 images
   Figures:           9 images

✅ Done!

💡 Text elements came from masked PDF JSON (clean, no tables/figures)
